In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
pd.options.display.float_format = '{:,.4f}'.format

In [ ]:
def mean_std(x):
    return f'{100*x.mean():.3f}+{100*x.std():.3f}'

def get_best_index(df, var_list, metric):
    df_mean = df.groupby(var_list)[metric].mean().sort_index()
    idx = df_mean.reset_index().groupby(['minor_ratio']).idxmax()[metric].dropna()
    return idx

def get_mean_with_given_index(df, var_list, metric, idx):
    df_mean = df.groupby(var_list)[metric].mean().sort_index()
    return df_mean.reset_index().loc[idx].set_index('minor_ratio')[metric]

def get_std_with_given_index(df, var_list, metric, idx):
    df_std = df.groupby(var_list)[metric].std().sort_index()
    return df_std.reset_index().loc[idx].set_index('minor_ratio')[metric]

def sort_and_query(df, sort_by, query):
    df_temp = df.sort_values(by=sort_by)
    df_temp = df_temp.query(query)
    return df_temp



def get_metric(df, module_name, idx_metric, metric, verbose=True):
    df_query = df.query('model == "convnext_t" and module_name == @module_name')

    ban_list = ['lr_history', 'data_seed', 'model_path', 'lr_history1', 'data_dir', 'epoch', 'max_epochs', 'train_loss', 'status', 'valid_valid_best_epoch', 'valid_valid_best_wc_epoch']
    var_list = (df_query.nunique() >= 2) & (df_query.nunique() <= 20)
    var_list = [i for i in df.columns[var_list] if 'acc' not in i and i not in ban_list]

    expected_length = 1

    if verbose:
        print(var_list)
        for i in var_list:
            print(i, df_query[i].nunique(), end=', ')
            expected_length *= df_query[i].nunique()
        print()
        print(f'Expected {expected_length}, got {len(df_query)}')

    idx = get_best_index(df_query, var_list, idx_metric)

    mean = get_mean_with_given_index(df_query, var_list, metric, idx)
    std = get_std_with_given_index(df_query, var_list, metric, idx)
    
    return mean, std

def get_wga(df, module_name):
    df_query = df.query('model == "convnext_t" and module_name == @module_name')

    ban_list = ['lr_history', 'data_seed', 'model_path', 'lr_history1', 'data_dir', 'epoch', 'max_epochs', 'train_loss']
    var_list = (df_query.nunique() >= 2) & (df_query.nunique() <= 20)
    var_list = [i for i in df.columns[var_list] if 'acc' not in i and i not in ban_list]

    expected_length = 1
    print(var_list)
    for i in var_list:
        print(i, df_query[i].nunique(), end=', ')
        expected_length *= df_query[i].nunique()
    print()
    print(f'Expected {expected_length}, got {len(df_query)}')

    idx = get_best_index(df_query, var_list, 'valid_valid_best_worst_acc')

    best_mean = get_mean_with_given_index(df_query, var_list, 'valid_test_worst_acc_by_best_val_worst', idx)
    best_std = get_std_with_given_index(df_query, var_list, 'valid_test_worst_acc_by_best_val_worst', idx)
    test_acc_mean = get_mean_with_given_index(df_query, var_list, 'valid_test_avg_acc_by_best_val_worst', idx)
    test_acc_std = get_std_with_given_index(df_query, var_list, 'valid_test_avg_acc_by_best_val_worst', idx)

    return best_mean, best_std, test_acc_mean, test_acc_std


def get_wga_by_wca(df, module_name):
    df_query = df.query('model == "convnext_t" and module_name == @module_name')

    ban_list = ['lr_history', 'data_seed', 'model_path', 'lr_history1', 'data_dir', 'epoch', 'max_epochs', 'train_loss', 'status', 'valid_valid_best_epoch', 'valid_valid_best_wc_epoch']
    var_list = (df_query.nunique() >= 2) & (df_query.nunique() <= 20)
    var_list = [i for i in df.columns[var_list] if 'acc' not in i and i not in ban_list]

    expected_length = 1
    print(var_list)
    for i in var_list:
        print(i, df_query[i].nunique(), end=', ')
        expected_length *= df_query[i].nunique()
    print()
    print(f'Expected {expected_length}, got {len(df_query)}')

    idx = get_best_index(df_query, var_list, 'valid_valid_worst_class_acc')

    best_mean = get_mean_with_given_index(df_query, var_list, 'valid_test_worst_acc_by_best_val_worst', idx)
    best_std = get_std_with_given_index(df_query, var_list, 'valid_test_worst_acc_by_best_val_worst', idx)
    test_acc_mean = get_mean_with_given_index(df_query, var_list, 'valid_test_avg_acc_by_best_val_cls_worst', idx)
    test_acc_std = get_std_with_given_index(df_query, var_list, 'valid_test_avg_acc_by_best_val_cls_worst', idx)

    return best_mean, best_std, test_acc_mean, test_acc_std

# 1. Get DataFrame from Neptune project

In [ ]:
old_project_list =['240411WaterBirds',
    '240406CelebaCollar',
    '240401CatDog']

new_project_list = ['240516CelebaCollar',
    '240516CatDog',
    '240516WaterBrids']

temp_dfs = {}

# Old files
for proj_name in old_project_list:
    df = pd.read_csv(f'results_viewer/exps/{proj_name}.csv')
    temp_dfs[proj_name] = df.query('module_name == "GDRO"')
    df_tmp = df.query('module_name == "GDRO"')

# New files
for proj_name in new_project_list:
    df = pd.read_csv(f'results_viewer_wca/exps/{proj_name}.csv')    

    idx = df['module_name'] == "MultiCGR"
    df.loc[idx, 'lamb_cs_list_1'] = df.loc[idx, 'lamb_cs_list'].apply(lambda x: float(x.split(',')[0]))
    df.loc[idx, 'lamb_cs_list_2'] = df.loc[idx, 'lamb_cs_list'].apply(lambda x: float(x.split(',')[1]))

    idx = df['lamb_cs_list_1'] == 0
    df.loc[idx, 'module_name'] = 'RRC-Signal'

    idx = df['lamb_cs_list_2'] == 0
    df.loc[idx, 'module_name'] = 'CGR'

    temp_dfs[proj_name] = df
    
temp_dfs = {
    'waterbirds': pd.concat([temp_dfs['240411WaterBirds'], temp_dfs['240516WaterBrids']]),
    'celeba_collar': pd.concat([temp_dfs['240406CelebaCollar'], temp_dfs['240516CelebaCollar']]),
    'catdog': pd.concat([temp_dfs['240401CatDog'], temp_dfs['240516CatDog']])
}

# Plot every results

In [ ]:
for proj_name, df in temp_dfs.items():
    print(proj_name)
    print(df['module_name'].value_counts())

    df_dict = {}
    module_list = list(df['module_name'].value_counts().index)
    for module_name in module_list:
        df_query = df.query('model == "convnext_t" and module_name == @module_name and dataset == @proj_name')

        ban_list = ['lr_history', 'data_seed', 'model_path', 'lr_history1', 'data_dir', 'epoch', 'max_epochs', 'train_loss']
        var_list = (df_query.nunique() >= 2) & (df_query.nunique() <= 20)
        var_list = [i for i in df.columns[var_list] if 'acc' not in i and i not in ban_list]

        print(module_name, var_list)

        df_query = df_query.sort_values(by=var_list)
        df_dict[module_name] = df_query.set_index(var_list)
    
        for var in var_list:
            print(var, df_query[var].unique())
        print()

        module_list = df['module_name'].unique()

    # get best_mean std and plot
    
    idx_metric_list = ['valid_valid_best_worst_acc', 'valid_valid_best_worst_acc', 'valid_valid_worst_class_acc', 'valid_valid_worst_class_acc']
    metric_list = ['valid_test_worst_acc_by_best_val_worst', 'valid_test_avg_acc_by_best_val_worst', 'valid_test_worst_acc_by_best_val_cls_worst', 'valid_test_avg_acc_by_best_val_cls_worst']

    for idx_metric, metric in zip(idx_metric_list, metric_list):

        best_mean_list = []
        best_std_list = []
        for module_name in module_list:
            best_mean, best_std = get_metric(df, module_name, idx_metric, metric)
            best_mean_list.append(best_mean)
            best_std_list.append(best_std)

        fig, ax = plt.subplots(1,1, figsize=(6,6))

        best_model_mean = pd.concat(best_mean_list, axis=1)
        best_model_mean.columns = module_list
        best_model_std = pd.concat(best_std_list, axis=1)
        best_model_std.columns = module_list

        # plot best model mean
        bars = best_model_mean.plot(kind='bar', yerr=best_model_std, ax=ax, capsize=5, legend=False, ylim=(0, 1))
        fig.legend(loc='upper center', bbox_to_anchor=(0.5, 0), ncol=5)
        ax.set_title(f'{proj_name} - {metric}')

        # plot mean and std at the same time
        for i, vs in enumerate(best_model_mean.values):
            for j, v in enumerate(vs):
                ax.text(i - 0.24 + j * (0.5/len(module_list)) , 0.05, str(f'{v:.3f}+-{best_model_std.values[i][j]:.3f}'), color='white', fontweight='bold', rotation=90)           
                        
        fig.savefig(f'figures/{proj_name}_{metric}.png', bbox_inches='tight')


# Lmabda - WGA graph

In [ ]:
def plot_lambda_wga_graph(df, ax, metric, var_list, x_label):
    df = df.groupby(var_list+[x_label])[metric].mean().reset_index().sort_values(by=x_label)
    df = df.groupby(var_list)
    for name, group in df:
        ax.plot(group[x_label], group[metric], label=name)
    ax.legend()
    ax.set_xlabel(x_label)
    ax.set_xscale('log')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} vs {x_label}')


In [ ]:
var_list = ['minor_ratio', 'learning_rate', 'batch_size_train']

# plot
fig, axes = plt.subplots(1,2, figsize=(12,6))

df_temp = df.query('module_name == "CGR"')
plot_lambda_wga_graph(df_temp, axes[0], 'valid_test_worst_acc_by_best_val_worst', var_list, 'lamb_cs_list_1')
plot_lambda_wga_graph(df_temp, axes[1], 'valid_valid_best_worst_acc', var_list, 'lamb_cs_list_1')

fig, axes = plt.subplots(1,2, figsize=(12,6))

df_temp = df.query('module_name == "RRC-Signal"')
plot_lambda_wga_graph(df_temp, axes[0], 'valid_test_worst_acc_by_best_val_worst', var_list, 'lamb_cs_list_2')
plot_lambda_wga_graph(df_temp, axes[1], 'valid_valid_best_worst_acc', var_list, 'lamb_cs_list_2')

# Making a table

In [ ]:
df_list = []
for proj_name, df in temp_dfs.items():
    df_dict = {}
    module_list = list(df['module_name'].value_counts().index)
    for module_name in module_list:
        df_query = df.query('model == "convnext_t" and module_name == @module_name and dataset == @proj_name')

        ban_list = ['lr_history', 'data_seed', 'model_path', 'lr_history1', 'data_dir', 'epoch', 'max_epochs', 'train_loss']
        var_list = (df_query.nunique() >= 2) & (df_query.nunique() <= 20)
        var_list = [i for i in df.columns[var_list] if 'acc' not in i and i not in ban_list]

        df_query = df_query.sort_values(by=var_list)
        df_dict[module_name] = df_query.set_index(var_list)
    
        module_list = df['module_name'].unique()

    # get best_mean std and plot
    
    idx_metric_list = ['valid_valid_best_worst_acc', 'valid_valid_best_worst_acc', 'valid_valid_worst_class_acc', 'valid_valid_worst_class_acc']
    metric_list = ['valid_test_worst_acc_by_best_val_worst', 'valid_test_avg_acc_by_best_val_worst', 'valid_test_worst_acc_by_best_val_cls_worst', 'valid_test_avg_acc_by_best_val_cls_worst']

    for idx_metric, metric in zip(idx_metric_list, metric_list):

        best_mean_list = []
        best_std_list = []
        for module_name in module_list:
            best_mean, best_std = get_metric(df, module_name, idx_metric, metric, verbose=False)
            best_mean_list.append(best_mean)
            best_std_list.append(best_std)

        best_model_mean = pd.concat(best_mean_list, axis=1)
        best_model_mean.columns = module_list
        best_model_std = pd.concat(best_std_list, axis=1)
        best_model_std.columns = module_list

        # Pivot 'minor_ratio' is applied in df_tmp. Undoing is needed
        df_tmp_mean = best_model_mean.stack().reset_index()
        df_tmp_std = best_model_std.stack().reset_index()
        
        # combine
        df_tmp = pd.concat([df_tmp_mean, df_tmp_std[0]], axis=1)
        df_tmp.columns = ['minor_ratio', 'module_name', 'mean', 'std']
        df_tmp['metric'] = metric
        df_tmp['dataset'] = proj_name

        df_list.append(df_tmp)


In [ ]:
dfs = pd.concat(df_list)

In [ ]:
# make mean and std columns as combined one
dfs['mean_std'] = dfs.apply(lambda x: f'{x["mean"]:.3f}+{x["std"]:.3f}', axis=1)
dfs.drop(columns=['mean', 'std'], inplace=True)

In [ ]:
for dataset in dfs['dataset'].unique():
    for metric in dfs['metric'].unique():
        # set table for latex 
        df_tmp = dfs.query('dataset == @dataset and metric == @metric')
        # df_tmp = df_tmp.pivot(index=['minor_ratio',], columns=['module_name'], values=['mean_std'])
        df_tmp = df_tmp.pivot(index=['minor_ratio',], columns=['module_name',], values=['mean_std'])
        print(df_tmp.to_latex())
        print()

In [ ]:
df_tmp = dfs.pivot(index=['dataset', 'minor_ratio',], columns=['metric', 'module_name', ], values=['mean_std'])
df_tmp